Loading the raw counts and metadata

In [1]:
import os
from pathlib import Path
os.chdir(Path.cwd().parent)
print(Path.cwd())

import pandas as pd

df_raw_cols = pd.read_csv('data/GSE189149_raw_counts_GRCh38.p13_NCBI.tsv.gz', sep='\t', index_col=0, nrows=0).columns.tolist()
metadata_csv = pd.read_csv('data/GSE189149_sample_metadata.csv')
print(df_raw_cols == metadata_csv['gsm_id'].tolist())

/Users/zaminsmac/Documents/Research/Cross-Cohort-Allergy-Biomarkers
True


In [6]:
#imports
import numpy as np
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference

In [7]:
#load counts + metadata
counts = pd.read_csv('data/GSE189149_raw_counts_GRCh38.p13_NCBI.tsv.gz',
                      sep='\t', index_col=0)
metadata = pd.read_csv('data/GSE189149_sample_metadata.csv', index_col='gsm_id')

print(counts.shape)
print(metadata.shape)

(39376, 48)
(48, 2)


In [8]:
#fix the orientation, then check alignment
counts = counts.T
print(counts.shape)  #should now say (48, 39376)

assert list(counts.index) == list(metadata.index), "order's off, fix before continuing"
print("alignment ok")

(48, 39376)
alignment ok


In [9]:
#drop genes with barely any counts across all samples
gene_totals = counts.sum(axis=0)  #total counts per gene, summed across all 48 samples
keep = gene_totals >= 10  #keep genes with at least 10 total reads overall

counts_filtered = counts.loc[:, keep]
print(f"{counts.shape[1]} genes before, {counts_filtered.shape[1]} after")

39376 genes before, 24745 after


In [10]:
#build the dataset + fit
inference = DefaultInference(n_cpus=4)  

dds = DeseqDataSet(
    counts=counts_filtered,
    metadata=metadata,
    design="~condition",
    inference=inference
)
dds.deseq2()

Using None as control genes, passed at DeseqDataSet initialization


/Users/zaminsmac/miniconda3/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
Fitting size factors...
... done in 0.04 seconds.

Fitting dispersions...
... done in 2.66 seconds.

Fitting dispersion trend curve...
... done in 0.26 seconds.

Fitting MAP dispersions...
... done in 3.51 seconds.

Fitting LFCs...
... done in 1.78 seconds.

Calculating cook's distance...
... done in 0.04 seconds.

Replacing 52 outlier genes.

Fitting dispersions...
... done in 0.04 seconds.

Fitting MAP dispersions...
... done in 0.02 seconds.

Fitting LFCs...
... done in 0.02 seconds.



In [12]:
#run the test
stat_res = DeseqStats(dds, contrast=["condition", "allergic", "control"], inference=inference)
stat_res.summary()
results_df = stat_res.results_df

Running Wald tests...


Log2 fold change & Wald test p-value: condition allergic vs control
              baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
GeneID                                                                        
100287102     1.886889        0.368569  0.403436  0.913574  0.360941  0.705736
653635      343.577440        0.025328  0.104049  0.243426  0.807675  0.936396
102466751     7.156201        0.184946  0.262274  0.705162  0.480710  0.788239
107985730     0.519661        0.537260  0.746529  0.719678  0.471723       NaN
100996442    17.308118        0.278813  0.170690  1.633451  0.102374  0.428643
...                ...             ...       ...       ...       ...       ...
4541       1814.715382       -0.020675  0.213033 -0.097050  0.922687  0.978370
4556        141.506925       -0.080796  0.185266 -0.436107  0.662759  0.879135
4519       6392.046730        0.098560  0.241322  0.408416  0.682968  0.888505
4576         25.652110       -0.073585  0.215511 -0.341443  0.7

... done in 0.59 seconds.



In [13]:
#isolate isg15
isg15_row = results_df.loc['9636']
print(isg15_row)

baseMean          3568.810290
log2FoldChange      -0.925921
lfcSE                0.401111
stat                -2.308393
pvalue               0.020977
padj                 0.245340
Name: 9636, dtype: float64


In [17]:
%pip install mygene
import mygene
mg = mygene.MyGeneInfo()

genes = ["ISG15", "SLC9A1", "EPHA6", "TMEM104", "RAB11FIP4", "HNRNPU", "KCTD2", "C18orf21", "CETN3"]
hits = mg.querymany(genes, scopes="symbol", fields="entrezgene", species="human", as_dataframe=True)

ids = hits["entrezgene"].dropna().astype(str)
id_map = hits["entrezgene"].dropna().astype(str)
lookup = results_df.loc[results_df.index.intersection(id_map.values),
                        ["baseMean", "log2FoldChange", "pvalue", "padj"]]
name_for = {num: sym for sym, num in id_map.items()}
lookup["gene"] = [name_for[i] for i in lookup.index]
lookup

Note: you may need to restart the kernel to use updated packages.


2 input query terms found no hit:	['TMEM104', 'C18orf21']


,baseMean,log2FoldChange,pvalue,padj,gene
GeneID,,,,,
9636,3568.810290,-0.925921,0.020977,0.245340,ISG15
6548,607.660466,0.016344,0.762784,0.918199,SLC9A1
3192,22458.152494,-0.013002,0.812359,0.938499,HNRNPU
1070,1060.991867,-0.181481,0.063852,0.362656,CETN3
84440,809.324715,0.341694,0.001032,0.111885,RAB11FIP4
23510,840.589287,-0.035387,0.500273,0.799015,KCTD2


In [19]:
extra = {"TMEM104": "54868", "C18orf21": "83608"}
counts_all = pd.read_csv("data/GSE189149_raw_counts_GRCh38.p13_NCBI.tsv.gz", sep="\t", index_col=0)
counts_all.index = counts_all.index.astype(str)

for gene, num in {**extra, "EPHA6": "285220"}.items():
    in_results = num in results_df.index
    total_reads = counts_all.loc[num].sum() if num in counts_all.index else None
    print(gene, num, "| in results:", in_results, "| total reads:", total_reads)
    if in_results:
        print(results_df.loc[num, ["baseMean", "log2FoldChange", "pvalue", "padj"]])

TMEM104 54868 | in results: True | total reads: 21850
baseMean          445.783350
log2FoldChange      0.097116
pvalue              0.092220
padj                0.410423
Name: 54868, dtype: float64
C18orf21 83608 | in results: True | total reads: 23464
baseMean          485.112846
log2FoldChange     -0.190247
pvalue              0.004689
padj                0.157132
Name: 83608, dtype: float64
EPHA6 285220 | in results: False | total reads: 5


In [20]:
import pandas as pd

# Probe → gene, with NCBI Gene IDs (verified on NCBI or via mygene)
genes = pd.DataFrame([
    ["cg08469540", "ISG15",     "",         "9636"],
    ["cg07202532", "RAB11FIP4", "",         "84440"],
    ["cg25316898", "RMP24",     "C18orf21", "83608"],
    ["cg02407224", "CETN3",     "",         "1070"],
    ["cg09821461", "SLC38A12",  "TMEM104",  "54868"],
    ["cg11305394", "KCTD2",     "",         "23510"],
    ["cg15076659", "SLC9A1",    "",         "6548"],
    ["cg01220651", "HNRNPU",    "",         "3192"],
    ["cg11090352", "EPHA6",     "",         "285220"],
    ["cg13858061", "",          "",         ""],
], columns=["probe", "gene", "former_name", "entrez"])

meth = pd.read_csv("data/../results/infant_fdr_hits.csv", index_col="probe")
counts_all = pd.read_csv("data/GSE189149_raw_counts_GRCh38.p13_NCBI.tsv.gz", sep="\t", index_col=0)
counts_all.index = counts_all.index.astype(str)

def rna(col):
    return [results_df.loc[e, col] if e in results_df.index else None for e in genes["entrez"]]

table = genes.copy()
table["infant_delta_beta"] = meth.loc[table["probe"], "delta_beta"].values
table["infant_adj_p"]      = meth.loc[table["probe"], "adj_p"].values
table["rna_log2FC"]        = rna("log2FoldChange")
table["rna_p"]             = rna("pvalue")
table["rna_padj_genomewide"] = rna("padj")
table["rna_total_reads"]   = [counts_all.loc[e].sum() if e in counts_all.index else None for e in genes["entrez"]]

n_tested = table["rna_p"].notna().sum()
threshold = 0.05 / n_tested
table["passes_bonferroni"] = table["rna_p"] < threshold

table["note"] = ""
table.loc[table["entrez"] == "", "note"] = "no annotated gene"
table.loc[(table["entrez"] != "") & table["rna_p"].isna(), "note"] = "filtered: low expression"

print(f"{n_tested} genes tested; Bonferroni threshold = {threshold:.5f}")
table.to_csv("results/infant_hits_rnaseq_lookup.csv", index=False)
table

8 genes tested; Bonferroni threshold = 0.00625


,probe,gene,former_name,entrez,infant_delta_beta,infant_adj_p,rna_log2FC,rna_p,rna_padj_genomewide,rna_total_reads,passes_bonferroni,note
0,cg08469540,ISG15,,9636,-0.028230,0.038277,-0.925921,0.020977,0.245340,170796.0,False,
1,cg07202532,RAB11FIP4,,84440,-0.034342,0.038277,0.341694,0.001032,0.111885,39922.0,True,
2,cg25316898,RMP24,C18orf21,83608,-0.012389,0.038277,-0.190247,0.004689,0.157132,23464.0,True,
3,cg02407224,CETN3,,1070,-0.028605,0.038277,-0.181481,0.063852,0.362656,50968.0,False,
4,cg09821461,SLC38A12,TMEM104,54868,-0.043788,0.049936,0.097116,0.092220,0.410423,21850.0,False,
5,cg11305394,KCTD2,,23510,-0.022900,0.038277,-0.035387,0.500273,0.799015,41089.0,False,
6,cg15076659,SLC9A1,,6548,-0.038097,0.038277,0.016344,0.762784,0.918199,29766.0,False,
7,cg01220651,HNRNPU,,3192,-0.032008,0.038277,-0.013002,0.812359,0.938499,1097493.0,False,
8,cg11090352,EPHA6,,285220,0.035173,0.048649,NaN,NaN,NaN,5.0,False,filtered: low expression
9,cg13858061,,,,0.019807,0.038277,NaN,NaN,NaN,NaN,False,no annotated gene


In [21]:
import subprocess, io
old = pd.read_csv(io.StringIO(subprocess.run(
    ["git", "show", "HEAD~1:data/deseq2_results_GSE189149.csv"],
    capture_output=True, text=True).stdout), index_col=0)
new = pd.read_csv("data/deseq2_results_GSE189149.csv", index_col=0)

shared = old.index.intersection(new.index)
print(len(old), len(new), len(shared))
print((old.loc[shared, "pvalue"] - new.loc[shared, "pvalue"]).abs().max())

24745 24745 24745
0.03634584917823719


In [22]:
diff = (old.loc[shared, "pvalue"] - new.loc[shared, "pvalue"]).abs()

print((diff > 0.001).sum(), "genes changed by more than 0.001")
print(diff.sort_values(ascending=False).head(10))
print(new.loc[diff.sort_values(ascending=False).head(10).index, "baseMean"])

old_sig = set(old.index[old["padj"] < 0.05])
new_sig = set(new.index[new["padj"] < 0.05])
print("FDR hits before:", len(old_sig), "after:", len(new_sig), "same set:", old_sig == new_sig)

import pydeseq2; print("pydeseq2 version:", pydeseq2.__version__)

60 genes changed by more than 0.001
GeneID
6431      0.036346
11100     0.036151
57403     0.028829
5037      0.028526
3032      0.027168
389677    0.025141
3059      0.022343
23598     0.022294
29102     0.018657
51742     0.016118
Name: pvalue, dtype: float64
GeneID
6431      4909.163452
11100     9405.590751
57403      804.713309
5037      5015.330833
3032      2206.052094
389677     787.197741
3059      6755.336572
23598      868.276395
29102     2078.500101
51742     1494.898892
Name: baseMean, dtype: float64
FDR hits before: 8 after: 8 same set: True
pydeseq2 version: 0.5.4


In [14]:
results_df.to_csv('data/deseq2_results_GSE189149.csv')

- RNA-seq differential expression (DESeq2) run on GSE189149 (48 adolescent samples, allergic vs. control), naive/activated CD4+ T cells.
- 39,376 genes before low-count filtering (≥10 total reads), 24,745 after.
- ISG15 (Entrez 9636): log2FC = -0.926, p = 0.021, padj = 0.245 (closely matches Imran et al.'s independently published result, log2FC = -0.931, p = 0.021, on the same samples)
- Full results saved to deseq2_results_GSE189149.csv, used downstream in notebook 10's cross-omics candidate ranking.

Feeds into notebook 10 (RNA-seq cross-validation of methylation candidates).